# Preparation

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/crowdflower-weather-twitter/sampleSubmission.csv
/kaggle/input/competitions/crowdflower-weather-twitter/variableNames.txt
/kaggle/input/competitions/crowdflower-weather-twitter/train.csv
/kaggle/input/competitions/crowdflower-weather-twitter/test.csv


In [2]:
# Import libraries
import keras
import re
import shutil
import sklearn
import string
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

# Import dependencies
from keras import layers
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# Set up charts and columns configuration
pd.options.display.max_rows = 2000
pd.options.display.max_columns = 500

2026-05-29 09:28:39.461947: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780046919.782163      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780046919.889812      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780046920.600258      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780046920.600334      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780046920.600338      16 computation_placer.cc:177] computation placer alr

# Examine the data

In [3]:
train = pd.read_csv('/kaggle/input/competitions/crowdflower-weather-twitter/train.csv')
test = pd.read_csv('/kaggle/input/competitions/crowdflower-weather-twitter/test.csv')

In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77946 entries, 0 to 77945
Data columns (total 28 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        77946 non-null  int64  
 1   tweet     77946 non-null  object 
 2   state     77946 non-null  object 
 3   location  66994 non-null  object 
 4   s1        77946 non-null  float64
 5   s2        77946 non-null  float64
 6   s3        77946 non-null  float64
 7   s4        77946 non-null  float64
 8   s5        77946 non-null  float64
 9   w1        77946 non-null  float64
 10  w2        77946 non-null  float64
 11  w3        77946 non-null  float64
 12  w4        77946 non-null  float64
 13  k1        77946 non-null  float64
 14  k2        77946 non-null  float64
 15  k3        77946 non-null  float64
 16  k4        77946 non-null  float64
 17  k5        77946 non-null  float64
 18  k6        77946 non-null  float64
 19  k7        77946 non-null  float64
 20  k8        77946 non-null  fl

In [5]:
# Display the number of indexes in the train dataset
for i in range(2):
    n = np.random.randint(77000)
    sample = train['tweet'][n]
    print('-' * 80)
    print(f'Index: {n}; \nSample: \n{sample}')

--------------------------------------------------------------------------------
Index: 9742; 
Sample: 
Great day in the gym! Getting lean and mean, now only if this weather would turn so I can get back outside!
--------------------------------------------------------------------------------
Index: 51806; 
Sample: 
@mention. Have a great Cinco de Mayo!  Enjoy your day, Sunshine! Isn't this your Fri?  ((HUGS))


In [6]:
test.info()
test.sample(2)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42157 entries, 0 to 42156
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        42157 non-null  int64 
 1   tweet     42157 non-null  object
 2   state     14323 non-null  object
 3   location  42144 non-null  object
dtypes: int64(1), object(3)
memory usage: 1.3+ MB


,id,tweet,state,location
5671,16174,I LOVE this weather!,NaN,brooklyn
26291,74834,Agreed. Unfortunately I've no choice. Burr! RT...,alabama,Birmingham AL


In [7]:
# Display the number of indexes in the test dataset
for i in range(2):
    n = np.random.randint(40000)
    sample = test['tweet'][n]
    print('-' * 80)
    print(f'Index: {n}; \nSample: \n{sample}')

--------------------------------------------------------------------------------
Index: 24787; 
Sample: 
Better tree care might have averted storm damage: Before Thursday's storm, there were "hundreds" of trees on the... {link}
--------------------------------------------------------------------------------
Index: 34204; 
Sample: 
Current Conditions: Fair, 60 FForecast: Sun - Sunny. High: 83 Low: 68 Mon - Sunny. High: 85 Low: 68Full Forecast at Yahoo! Weather (p...


# Sentence standardization

In [8]:
# Define configurations
def custom_standardization(sentence):
    sample = tf.strings.lower(sentence)
    stripped_html = tf.strings.regex_replace(sample, 'link', '')
    return tf.strings.regex_replace(stripped_html, '[%s]'%re.escape(string.punctuation), '')

max_features = 10000
sequence_length = 250

vectorize_layer = layers.TextVectorization(
    standardize=custom_standardization,
    max_tokens=max_features,
    output_mode='int',
    output_sequence_length=sequence_length
)
vectorize_layer.adapt(train['tweet'])

for i in range(1):
    sample = np.random.randint(77000)
    print(f"Before standardization: \n{train['tweet'][sample]}")
    print('-' * 80)
    print(f"After standardization: \n{custom_standardization(train['tweet'][sample])}")
    print('-' * 80)
    print(f"After vectorization: \n{vectorize_layer(train['tweet'][sample])}")
    print('*' * 80)

2026-05-29 09:29:01.449772: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Before standardization: 
What a great weekend. Fabulous weather. And we've seen both sets of parents. Yet it's still been great!
--------------------------------------------------------------------------------
After standardization: 
b'what a great weekend fabulous weather and weve seen both sets of parents yet its still been great'
--------------------------------------------------------------------------------
After vectorization: 
[  74    7   67  111 1544    3    8  647  657  887 4254   14 1509  357
   11  108   95   67    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0

# Model training

In [9]:
x = train.iloc[:, 1:2].values
X = np.array(vectorize_layer(x))
X.shape

(77946, 250)

In [10]:
Y = np.array(train.iloc[:, 4:])
Y.shape

(77946, 24)

In [11]:
a = test.iloc[:, 1:2].values
A = np.array(vectorize_layer(a))
A.shape

(42157, 250)

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3)
print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print('-' * 50)
print(f'X_test: {X_test.shape}, y_test: {y_test.shape}')

X_train: (54562, 250), y_train: (54562, 24)
--------------------------------------------------
X_test: (23384, 250), y_test: (23384, 24)


In [13]:
# Train the model using Lasso function
from sklearn.linear_model import Lasso

model = Lasso()

_ = model.fit(X_train, y_train)
y_pred = model.predict(X_test)
np.sqrt(mean_squared_error(y_test, y_pred))

np.float64(0.24763717967532858)

In [14]:
# Predict the model
predict = model.predict(A)
predict

array([[0.0624666 , 0.27102109, 0.27987251, ..., 0.16561135, 0.01003075,
        0.06849934],
       [0.02532668, 0.12189266, 0.52994089, ..., 0.09520182, 0.02663289,
        0.11493754],
       [0.04435019, 0.16860423, 0.45713464, ..., 0.10722175, 0.01646926,
        0.12716009],
       ...,
       [0.04422508, 0.23419374, 0.31346373, ..., 0.15835799, 0.02911162,
        0.05112622],
       [0.05576407, 0.23810597, 0.31591205, ..., 0.14706924, 0.01478144,
        0.06045137],
       [0.06260303, 0.27836351, 0.26085375, ..., 0.17194788, 0.01016896,
        0.05343303]], shape=(42157, 24))

# File submission

In [15]:
# Create a submission file
submission = pd.read_csv('/kaggle/input/competitions/crowdflower-weather-twitter/sampleSubmission.csv')
ids = test['id']
label = pd.DataFrame(predict, columns=submission.iloc[:, 1:].columns)
submission = pd.concat([ids, label], axis=1)
submission.to_csv("submission.csv", index=False)

In [16]:
submission

,id,s1,s2,s3,s4,s5,w1,w2,w3,w4,k1,k2,k3,k4,k5,k6,k7,k8,k9,k10,k11,k12,k13,k14,k15
0,4,0.062467,0.271021,0.279873,0.254289,0.130382,0.772280,0.078069,0.123079,0.031260,0.021131,0.117543,0.008222,0.137046,0.052469,0.001821,0.285195,0.003425,0.076994,0.101223,0.030714,0.105675,0.165611,0.010031,0.068499
1,5,0.025327,0.121893,0.529941,0.099786,0.226754,0.672421,0.093371,0.141118,0.085826,0.023042,0.059875,0.008741,0.069114,0.086975,0.001821,0.271952,0.003425,0.091153,0.097098,0.041242,0.241680,0.095202,0.026633,0.114938
2,7,0.044350,0.168604,0.457135,0.136305,0.195243,0.721921,0.074660,0.151346,0.054493,0.024304,0.086559,0.009597,0.104280,0.117396,0.001821,0.291615,0.003425,0.094893,0.078497,0.032962,0.152425,0.107222,0.016469,0.127160
3,8,0.049522,0.209523,0.387514,0.190480,0.162792,0.743949,0.081230,0.130828,0.047145,0.021465,0.098284,0.008241,0.112180,0.090782,0.001821,0.288014,0.003425,0.084105,0.093562,0.033684,0.135950,0.135112,0.015731,0.096553
4,12,0.052120,0.232687,0.326569,0.224997,0.163945,0.728423,0.087243,0.129944,0.056060,0.026675,0.093040,0.007531,0.122339,0.041906,0.001821,0.289019,0.003425,0.076961,0.098074,0.034626,0.160752,0.145609,0.017428,0.056257
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42152,120094,0.028830,0.121594,0.548836,0.112895,0.190651,0.701056,0.087839,0.138912,0.068930,0.021735,0.064856,0.008726,0.079066,0.137109,0.001821,0.273207,0.003425,0.100869,0.084812,0.034269,0.211951,0.100582,0.021793,0.132381
42153,120096,0.064410,0.294040,0.237398,0.283019,0.118404,0.778637,0.081621,0.114704,0.029501,0.022829,0.119033,0.007281,0.142187,0.026539,0.001821,0.283289,0.003425,0.069469,0.106756,0.030931,0.113030,0.176956,0.010203,0.042354
42154,120099,0.044225,0.234194,0.313464,0.225764,0.182249,0.704270,0.112446,0.117541,0.064231,0.024700,0.089458,0.008742,0.116951,0.026820,0.001821,0.233775,0.003425,0.078725,0.116720,0.034588,0.187193,0.158358,0.029112,0.051126
42155,120101,0.055764,0.238106,0.315912,0.211858,0.177946,0.728327,0.086128,0.132869,0.051862,0.022259,0.097478,0.007531,0.115434,0.033738,0.001821,0.278152,0.003425,0.073530,0.105477,0.041774,0.157047,0.147069,0.014781,0.060451


In [17]:
print("Successfully saved as CSV file.")

Successfully saved as CSV file.
